In [2]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import os
import optuna
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, RobustScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate, KFold, GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error, mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
import warnings


In [ ]:
#MARKET MAKER 1-9 LOOSE - IGNORE THIS IS JUST CURSORY

def run_adaptive_market_maker(data_folder="hackathon_data", num_rounds=9):
    results = []

    for round_num in range(1, num_rounds + 1):
        print(f"\n--- Processing Round {round_num} ---")
        
        train_path = os.path.join(data_folder, f"stock_{round_num}_train.csv")
        test_path = os.path.join(data_folder, f"stock_{round_num}_test.csv")
        
        try:
            train = pd.read_csv(train_path)
            test = pd.read_csv(test_path)
        except FileNotFoundError:
            print(f"Data for round {round_num} not found.")
            continue

        X = train.drop("target", axis=1)
        y = train["target"]
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        test_scaled = scaler.transform(test)

        X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

        #AMODEL SELECTION
        num_rows = train.shape[0]
        print(f"Dataset size: {num_rows} rows. ", end="")
        
        if num_rows < 500:
            print("Using Ridge Regression (Small Data Strategy)")
            # Alpha acts as a dampener to prevent overfitting on small data
            model = Ridge(alpha=1.0, random_state=42) 
        else:
            print("Using XGBoost (Large Data Strategy)")
            model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)

        # Train and validate to find our error margin
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        model_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        print(f"Validation RMSE: {model_rmse:.4f}")

        # Retrain on ALL data for the final prediction
        model.fit(X_scaled, y)
        mid_pred = model.predict(test_scaled)[0]

        spread_multiplier = 1.0 
        
        bid_pred = mid_pred - (spread_multiplier * model_rmse)
        ask_pred = mid_pred + (spread_multiplier * model_rmse)
        spread = ask_pred - bid_pred

        print(f"Predicted Target (Mid): {mid_pred:.2f}")
        print(f"Bid Price: {bid_pred:.2f}")
        print(f"Ask Price: {ask_pred:.2f}")
        print(f"Spread: {spread:.2f}")

        results.append({
            'Round': round_num,
            'Mid': mid_pred,
            'Bid': bid_pred,
            'Ask': ask_pred,
            'Spread': spread
        })

    return pd.DataFrame(results)

run_adaptive_market_maker()

In [ ]:
#EXPLORATORY ANALYSIS

def run_hackathon_eda(data_folder="hackathon_data", num_stocks=9):
    print("=== QMML Hackathon: Exploratory Data Analysis ===")
    
    # 1. Setup the 3x3 Grid for Target Histograms
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    axes = axes.flatten()
    
    # 2. Loop through all 9 stocks
    for i in range(1, num_stocks + 1):
        file_path = os.path.join(data_folder, f"stock_{i}_train.csv")
        
        # Check if the file exists before crashing
        if not os.path.exists(file_path):
            axes[i-1].set_title(f"Stock {i} (File Not Found)", color='red')
            continue
            
        df = pd.read_csv(file_path)
        
        # --- PART A: TARGET DISTRIBUTION (THE SKEW TRAP) ---
        target = df['target']
        skewness = target.skew()
        
        # Plot the histogram
        sns.histplot(target, kde=True, ax=axes[i-1], color='steelblue', bins=30)
        
        # If skewness > 1.0 or < -1.0, it's highly skewed and dangerous
        title_color = 'red' if abs(skewness) > 1.0 else 'black'
        axes[i-1].set_title(f"Stock {i} Target | Skew: {skewness:.2f}", color=title_color, fontweight='bold')
        axes[i-1].set_xlabel("Price")
        
        # --- PART B: FEATURE OUTLIERS (THE SCALING TRAP) ---
        features = df.drop('target', axis=1)
        
        # Calculate summary statistics for the features
        feature_summary = pd.DataFrame({
            'Min': features.min(),
            'Max': features.max(),
            'Mean': features.mean(),
            'Range': features.max() - features.min()
        })
        
        print(f"\n--- STOCK {i} EDA REPORT ---")
        print(f"Dataset Size: {len(df)} rows, {features.shape[1]} features.")
        
        if abs(skewness) > 1.0:
            print("⚠️ WARNING: Target is highly skewed! Consider using Log Transformation (np.log1p) before training.")
        else:
            print("✅ Target distribution is relatively normal.")
            
        # Check for extreme feature ranges (Assuming standard ranges should be under 50 based on Stock 1/2)
        outlier_columns = feature_summary[feature_summary['Range'] > 50]
        
        if not outlier_columns.empty:
            print("⚠️ WARNING: Extreme feature outliers detected. Standard scaling will fail!")
            print("Use RobustScaler instead of StandardScaler for these columns:")
            print(outlier_columns[['Min', 'Max', 'Range']].to_string())
        else:
            print("✅ Feature scales are clean and contained (No extreme outliers detected).")

    # 3. Finalize and show the plots
    plt.tight_layout()
    plt.savefig("all_9_stocks_histograms.png", dpi=300)
    print("\n[!] Histogram grid saved as 'all_9_stocks_histograms.png'. Open it to visually inspect the distributions.")
    # plt.show() # Uncomment if running in a Jupyter Notebook

# Make sure your files are in the 'hackathon_data' folder, then run:
run_hackathon_eda(data_folder="hackathon_data", num_stocks=9)

In [ ]:
#HYPERPARAMETER TUNING
import pandas as pd
import numpy as np
import optuna
import warnings
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate

# Keep terminal clean
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def run_ultimate_tuning_lab(stock_num, n_trials=1000):
    try:
        train = pd.read_csv(f"hackathon_data/stock_{stock_num}_train.csv")
    except FileNotFoundError:
        print(f"[!] Data for Stock {stock_num} not found. Ensure the path is correct.")
        return None

    X_full = train.drop("target", axis=1)
    y_full = train["target"]
    num_rows = len(X_full)
    apply_log_transform = abs(y_full.skew()) > 1.0

    def objective(trial):
        # ==========================================
        # 🧬 THE GENETIC FEATURE MUTATOR 🧬
        # ==========================================
        X_mutated = X_full.copy()
        base_cols = list(X_mutated.columns)
        
        # 1. Dynamic Interactions (Up to 3 combos)
        for i in range(3):
            if trial.suggest_categorical(f"build_interaction_{i}", [True, False]):
                c1 = trial.suggest_categorical(f"inter_{i}_c1", base_cols)
                c2 = trial.suggest_categorical(f"inter_{i}_c2", base_cols)
                op = trial.suggest_categorical(f"inter_{i}_op", ["add", "sub", "mult", "div"])
                
                col_name = f"{c1}_{op}_{c2}"
                if op == "add": X_mutated[col_name] = X_mutated[c1] + X_mutated[c2]
                elif op == "sub": X_mutated[col_name] = X_mutated[c1] - X_mutated[c2]
                elif op == "mult": X_mutated[col_name] = X_mutated[c1] * X_mutated[c2]
                elif op == "div": X_mutated[col_name] = X_mutated[c1] / (X_mutated[c2] + 1e-9)

        # 2. Non-Linear Expansions (Squares)
        for col in base_cols:
            if trial.suggest_categorical(f"square_{col}", [True, False]):
                X_mutated[f"{col}_squared"] = X_mutated[col] ** 2
                
        # 3. Automated Dropper
        cols_to_drop = []
        for col in X_mutated.columns:
            if trial.suggest_float(f"drop_prob_{col}", 0.0, 1.0) > 0.8:
                cols_to_drop.append(col)
                
        if len(cols_to_drop) < len(X_mutated.columns):
            X_mutated = X_mutated.drop(columns=cols_to_drop)

        # ==========================================
        # 🤖 THE MODEL BUILDER 🤖
        # ==========================================
        scaler_str = trial.suggest_categorical("scaler", ["Standard", "Robust"])
        scaler = StandardScaler() if scaler_str == "Standard" else RobustScaler()
        
        # --- SMALL DATA SAFEGUARD ---
        if num_rows < 500:
            m_type = trial.suggest_categorical("model", ["Ridge", "Lasso"])
        else:
            m_type = trial.suggest_categorical("model", ["Ridge", "Lasso", "HistGBM", "XGBoost", "LightGBM"])
        
        if m_type == "Ridge":
            base_model = Ridge(alpha=trial.suggest_float("alpha", 1e-3, 10.0, log=True), random_state=42)
        elif m_type == "Lasso":
            base_model = Lasso(alpha=trial.suggest_float("alpha", 1e-4, 1.0, log=True), random_state=42, max_iter=2000)
        elif m_type == "XGBoost":
            base_model = xgb.XGBRegressor(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                subsample=trial.suggest_float("subsample", 0.5, 1.0),
                random_state=42, n_jobs=-1,
                tree_method="hist",  # <-- GPU UPGRADE
                device="cuda"        # <-- GPU UPGRADE
            )
        elif m_type == "LightGBM":
            base_model = lgb.LGBMRegressor(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                num_leaves=trial.suggest_int("num_leaves", 20, 40),
                random_state=42, n_jobs=-1, verbose=-1,
                device_type="gpu"    # <-- GPU UPGRADE
            )
        else:
            base_model = HistGradientBoostingRegressor(
                max_iter=trial.suggest_int("max_iter", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                random_state=42
            )

        model = TransformedTargetRegressor(regressor=base_model, func=np.log1p, inverse_func=np.expm1) if apply_log_transform else base_model
        pipeline = Pipeline([('scaler', scaler), ('model', model)])

        # ==========================================
        # 📊 TIME-SERIES VALIDATION (No Leakage)
        # ==========================================
        tscv = TimeSeriesSplit(n_splits=5)
        scores = cross_validate(pipeline, X_mutated, y_full, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
        return -scores['test_score'].mean()

    # --- REPRODUCIBILITY LOCK ---
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials)
    
    return study.best_params

if __name__ == "__main__":
    TARGET_STOCK = 1  # Change this number to test different stocks
    
    print(f"🚀 INITIATING QUANT LAB: STOCK {TARGET_STOCK} (1000 TRIALS) 🚀")
    
    best_params = run_ultimate_tuning_lab(TARGET_STOCK, n_trials=1000)
    
    if best_params:
        print("\n" + "="*50)
        print(f"🏆 BEST PARAMETERS FOUND FOR STOCK {TARGET_STOCK} 🏆")
        print("="*50)
        for key, value in best_params.items():
            print(f"  - {key}: {value}")
        print("="*50)
        print("\n[✔] Tuning Complete. Copy these into your execution script or manually test them.")

KeyboardInterrupt: 

In [ ]:
#STOCK 1

# ==========================================
# A. LOAD DATA (Change stock_num for each round)
# ==========================================
stock_num = 1
train = pd.read_csv(f"hackathon_data/stock_{stock_num}_train.csv")
test = pd.read_csv(f"hackathon_data/stock_{stock_num}_test.csv")

X = train.drop("target", axis=1)
y = train["target"]

# ==========================================
# B. THE MODULAR PIPELINE (Optuna Tuned)
# ==========================================

# 1. Update the Scaler
scaler = RobustScaler()

# 2. Update the Model
tree_model = xgb.XGBRegressor(
    n_estimators=492,
    learning_rate=0.020053478210758077,
    max_depth=4,
    subsample=0.5480678723641842,
    random_state=42,
    n_jobs=-1
)

# 3. Combine them into your final pipeline
model = Pipeline([
    ('scaler', scaler),
    ('model', tree_model)
])

# ==========================================
# C. RISK ANALYSIS (5-Fold Cross Validation)
# ==========================================
print(f"--- Running Risk Diagnostics for Stock {stock_num} ---")
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error'
}

cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)

mean_mae = (-cv_results['test_mae']).mean()
mean_rmse = (-cv_results['test_rmse']).mean()
cv_volatility = (-cv_results['test_mae']).std() / mean_mae

outlier_ratio = mean_rmse / mean_mae
num_rows = len(train)

print(f"Dataset Size:  {num_rows} rows")
print(f"Average MAE:   {mean_mae:.4f}")
print(f"Average RMSE:  {mean_rmse:.4f}")
print(f"Outlier Ratio: {outlier_ratio:.4f}")
print(f"CV Volatility: {cv_volatility:.2%}")

# ==========================================
# D. THE DECISION ENGINE (Automatic Spread Sizing)
# ==========================================
# ==========================================
# D. THE DYNAMIC DECISION ENGINE
# ==========================================
print("\n--- Strategy Engine ---")

# YOUR STRATEGY DIAL (1 = Safest, 10 = Most Aggressive)
# Set this based on how badly you want to win the Market Maker spot for this round.
target_aggression = 8

# 1. The Hard Circuit Breakers (Absolute Defense)
if num_rows < 500:
    print("Trigger: Small Dataset. CIRCUIT BREAKER ACTIVATED.")
    spread_margin = mean_rmse * 1.0  
    
elif outlier_ratio > 1.30 or cv_volatility > 0.15:
    print("Trigger: High Tail Risk. CIRCUIT BREAKER ACTIVATED.")
    spread_margin = mean_rmse * 1.0  

# 2. Continuous Risk Scaling (Fluid Aggression)
else:
    print("Trigger: Model Stable. Calculating dynamic confidence spread...")
    
    # Scale aggression from 1-10 into a base math multiplier (10 -> 0.1, 1 -> 1.0)
    base_multiplier = (11 - target_aggression) / 10.0
    
    # We penalize the multiplier based on how volatile the cross-validation was.
    # If volatility is 1% (0.01), it barely changes the multiplier.
    # If volatility is 10% (0.10), it significantly widens the spread.
    volatility_penalty = cv_volatility * 5.0 
    
    # Calculate final dynamic multiplier (Capped at a minimum of 0.05 so spread is never exactly zero)
    dynamic_multiplier = max(0.05, base_multiplier + volatility_penalty)
    
    # Apply the dynamic multiplier to the MAE
    spread_margin = mean_mae * dynamic_multiplier
    
    print(f"  -> User Aggression Level: {target_aggression}/10")
    print(f"  -> Model Volatility Penalty: +{volatility_penalty:.3f}")
    print(f"  -> Final Applied MAE Multiplier: {dynamic_multiplier:.3f}")

# ==========================================
# E. FINAL PREDICTION & SUBMISSION NUMBERS
# ==========================================
# Retrain the pipeline on ALL available training data for maximum accuracy
model.fit(X, y)

# Predict the single missing target value in the test set
mid_pred = model.predict(test)[0]

# Calculate the final Bid (Buy) and Ask (Sell) prices
bid_pred = mid_pred - spread_margin
ask_pred = mid_pred + spread_margin
total_spread = ask_pred - bid_pred

print(f"\n=====================================")
print(f"🏆 FINAL SUBMISSION: STOCK {stock_num} 🏆")
print(f"=====================================")
print(f"Predicted Mid: {mid_pred:.4f}")
print(f"Bid Price:     {bid_pred:.4f}")
print(f"Ask Price:     {ask_pred:.4f}")
print(f"Total Spread:  {total_spread:.4f}")
print(f"=====================================")

In [ ]:
#STOCK 2

# ==========================================
# A. LOAD DATA (Change stock_num for each round)
# ==========================================
stock_num = 2
train = pd.read_csv(f"hackathon_data/stock_{stock_num}_train.csv")
test = pd.read_csv(f"hackathon_data/stock_{stock_num}_test.csv")

X = train.drop("target", axis=1)
y = train["target"]

# ==========================================
# B. THE MODULAR PIPELINE (Swap this based on Optuna results)
# ==========================================


# 1. Update the Scaler
scaler = RobustScaler()

# 2. Update the Model
tree_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.045663081372688076,
    max_depth=3,
    subsample=0.5284498661704208,
    random_state=42,
    n_jobs=-1
)

# 3. Combine them into your final pipeline
model = Pipeline([
    ('scaler', scaler),
    ('model', tree_model)
])


# ==========================================
# C. RISK ANALYSIS (5-Fold Cross Validation)
# ==========================================
print(f"--- Running Risk Diagnostics for Stock {stock_num} ---")
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error'
}

cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)

mean_mae = (-cv_results['test_mae']).mean()
mean_rmse = (-cv_results['test_rmse']).mean()
cv_volatility = (-cv_results['test_mae']).std() / mean_mae

outlier_ratio = mean_rmse / mean_mae
num_rows = len(train)

print(f"Dataset Size:  {num_rows} rows")
print(f"Average MAE:   {mean_mae:.4f}")
print(f"Average RMSE:  {mean_rmse:.4f}")
print(f"Outlier Ratio: {outlier_ratio:.4f}")
print(f"CV Volatility: {cv_volatility:.2%}")

# ==========================================
# D. THE DYNAMIC DECISION ENGINE
# ==========================================
print("\n--- Strategy Engine ---")

# YOUR STRATEGY DIAL (1 = Safest, 10 = Most Aggressive)
# Set this based on how badly you want to win the Market Maker spot for this round.
target_aggression = 5  

# 1. The Hard Circuit Breakers (Absolute Defense)
if num_rows < 500:
    print("Trigger: Small Dataset. CIRCUIT BREAKER ACTIVATED.")
    spread_margin = mean_rmse * 1.0  
    
elif outlier_ratio > 1.30 or cv_volatility > 0.15:
    print("Trigger: High Tail Risk. CIRCUIT BREAKER ACTIVATED.")
    spread_margin = mean_rmse * 1.0  

# 2. Continuous Risk Scaling (Fluid Aggression)
else:
    print("Trigger: Model Stable. Calculating dynamic confidence spread...")
    
    # Scale aggression from 1-10 into a base math multiplier (10 -> 0.1, 1 -> 1.0)
    base_multiplier = (11 - target_aggression) / 10.0
    
    # We penalize the multiplier based on how volatile the cross-validation was.
    # If volatility is 1% (0.01), it barely changes the multiplier.
    # If volatility is 10% (0.10), it significantly widens the spread.
    volatility_penalty = cv_volatility * 5.0 
    
    # Calculate final dynamic multiplier (Capped at a minimum of 0.05 so spread is never exactly zero)
    dynamic_multiplier = max(0.05, base_multiplier + volatility_penalty)
    
    # Apply the dynamic multiplier to the MAE
    spread_margin = mean_mae * dynamic_multiplier
    
    print(f"  -> User Aggression Level: {target_aggression}/10")
    print(f"  -> Model Volatility Penalty: +{volatility_penalty:.3f}")
    print(f"  -> Final Applied MAE Multiplier: {dynamic_multiplier:.3f}")

# ==========================================
# E. FINAL PREDICTION & SUBMISSION NUMBERS
# ==========================================
# Retrain the pipeline on ALL available training data for maximum accuracy
model.fit(X, y)

# Predict the single missing target value in the test set
mid_pred = model.predict(test)[0]

# Calculate the final Bid (Buy) and Ask (Sell) prices
bid_pred = mid_pred - spread_margin
ask_pred = mid_pred + spread_margin
total_spread = ask_pred - bid_pred

print(f"\n=====================================")
print(f"🏆 FINAL SUBMISSION: STOCK {stock_num} 🏆")
print(f"=====================================")
print(f"Predicted Mid: {mid_pred:.4f}")
print(f"Bid Price:     {bid_pred:.4f}")
print(f"Ask Price:     {ask_pred:.4f}")
print(f"Total Spread:  {total_spread:.4f}")
print(f"=====================================")

In [ ]:
#STOCK 3

# ==========================================
# A. LOAD DATA (Change stock_num for each round)
# ==========================================
stock_num = 3
train = pd.read_csv(f"hackathon_data/stock_{stock_num}_train.csv")
test = pd.read_csv(f"hackathon_data/stock_{stock_num}_test.csv")

X = train.drop("target", axis=1)
y = train["target"]

# ==========================================
# B. THE MODULAR PIPELINE (Swap this based on Optuna results)
# ==========================================

# 1. Update the Scaler
# Optuna chose the RobustScaler to handle underlying outliers.
scaler = RobustScaler()

# 2. Update the Base Model
# Optuna selected Lasso with a very low penalty.
# We hardcode random_state=42 and max_iter=2000 for stability.
linear_model = Lasso(
    alpha=0.0010026444968410334, 
    random_state=42, 
    max_iter=2000
)

# 3. Combine them into your final pipeline
# No TransformedTargetRegressor is needed because the skew was low!
model = Pipeline([
    ('scaler', scaler),
    ('model', linear_model)
])

# ==========================================
# C. RISK ANALYSIS (5-Fold Cross Validation)
# ==========================================
print(f"--- Running Risk Diagnostics for Stock {stock_num} ---")
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error'
}

cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)

mean_mae = (-cv_results['test_mae']).mean()
mean_rmse = (-cv_results['test_rmse']).mean()
cv_volatility = (-cv_results['test_mae']).std() / mean_mae

outlier_ratio = mean_rmse / mean_mae
num_rows = len(train)

print(f"Dataset Size:  {num_rows} rows")
print(f"Average MAE:   {mean_mae:.4f}")
print(f"Average RMSE:  {mean_rmse:.4f}")
print(f"Outlier Ratio: {outlier_ratio:.4f}")
print(f"CV Volatility: {cv_volatility:.2%}")

# ==========================================
# D. THE DYNAMIC DECISION ENGINE
# ==========================================
print("\n--- Strategy Engine ---")

# YOUR STRATEGY DIAL (1 = Safest, 10 = Most Aggressive)
# Set this based on how badly you want to win the Market Maker spot for this round.
target_aggression = 5  

# 1. The Hard Circuit Breakers (Absolute Defense)
if num_rows < 500:
    print("Trigger: Small Dataset. CIRCUIT BREAKER ACTIVATED.")
    spread_margin = mean_rmse * 1.0  
    
elif outlier_ratio > 1.30 or cv_volatility > 0.15:
    print("Trigger: High Tail Risk. CIRCUIT BREAKER ACTIVATED.")
    spread_margin = mean_rmse * 1.0  

# 2. Continuous Risk Scaling (Fluid Aggression)
else:
    print("Trigger: Model Stable. Calculating dynamic confidence spread...")
    
    # Scale aggression from 1-10 into a base math multiplier (10 -> 0.1, 1 -> 1.0)
    base_multiplier = (11 - target_aggression) / 10.0
    
    # We penalize the multiplier based on how volatile the cross-validation was.
    # If volatility is 1% (0.01), it barely changes the multiplier.
    # If volatility is 10% (0.10), it significantly widens the spread.
    volatility_penalty = cv_volatility * 5.0 
    
    # Calculate final dynamic multiplier (Capped at a minimum of 0.05 so spread is never exactly zero)
    dynamic_multiplier = max(0.05, base_multiplier + volatility_penalty)
    
    # Apply the dynamic multiplier to the MAE
    spread_margin = mean_mae * dynamic_multiplier
    
    print(f"  -> User Aggression Level: {target_aggression}/10")
    print(f"  -> Model Volatility Penalty: +{volatility_penalty:.3f}")
    print(f"  -> Final Applied MAE Multiplier: {dynamic_multiplier:.3f}")

# ==========================================
# E. FINAL PREDICTION & SUBMISSION NUMBERS
# ==========================================
# Retrain the pipeline on ALL available training data for maximum accuracy
model.fit(X, y)

# Predict the single missing target value in the test set
mid_pred = model.predict(test)[0]

# Calculate the final Bid (Buy) and Ask (Sell) prices
bid_pred = mid_pred - spread_margin
ask_pred = mid_pred + spread_margin
total_spread = ask_pred - bid_pred

print(f"\n=====================================")
print(f"🏆 FINAL SUBMISSION: STOCK {stock_num} 🏆")
print(f"=====================================")
print(f"Predicted Mid: {mid_pred:.4f}")
print(f"Bid Price:     {bid_pred:.4f}")
print(f"Ask Price:     {ask_pred:.4f}")
print(f"Total Spread:  {total_spread:.4f}")
print(f"=====================================")